# Constant-Velocity Baseline

This notebook evaluates a simple hand-written maneuver baseline on the exact same split used by the learned models.


## Run Safety

Before running this notebook, restart the kernel.

Why this matters:
- previous model runs can leave tensors and cached arrays in memory
- old variables can leak into the current run and make comparisons unfair
- we want every notebook to save a clean, reproducible result

Recommended workflow:
1. Restart kernel
2. Run all cells from top to bottom
3. Let the notebook save its outputs before closing it


In [1]:
# Memory cleanup before starting this notebook.
import gc

gc.collect()
try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
except Exception:
    pass

print('Kernel memory cleanup complete. Start the notebook from here after a restart.')


Kernel memory cleanup complete. Start the notebook from here after a restart.


In [2]:
# Standard-library imports used throughout the evaluation notebooks.
import json
import math
import random
import sys
from collections import Counter
from dataclasses import dataclass
from functools import lru_cache
from pathlib import Path
from datetime import datetime
from IPython.display import display

# Core scientific / ML stack.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, Subset

# ROC / PR curves are useful for classification comparison if sklearn is available.
try:
    from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score
    from sklearn.preprocessing import label_binarize
    SKLEARN_AVAILABLE = True
except Exception:
    SKLEARN_AVAILABLE = False

# Resolve the project root robustly no matter where Jupyter was launched from.
SEARCH_ROOT = Path.cwd().resolve()
PROJECT_ROOT = SEARCH_ROOT
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / '04_model_evaluation').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

EVAL_ROOT = PROJECT_ROOT / '04_model_evaluation'
THESIS_ROOT = EVAL_ROOT
NOTEBOOK_ROOT = EVAL_ROOT / 'notebooks'
RESULTS_ROOT = EVAL_ROOT / 'results'
WEIGHTS_ROOT = EVAL_ROOT / 'model_weights'
SPLITS_ROOT = EVAL_ROOT / 'splits'
PLOTS_ROOT = EVAL_ROOT / 'plots'
if str(NOTEBOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_ROOT))
ORIGINAL_DATASET_ROOT = PROJECT_ROOT / '03_dataset' / 'hybrid_maneuvers_dataset'
PREFERRED_DATASET_ROOT = ORIGINAL_DATASET_ROOT
DATASET_ROOT = PREFERRED_DATASET_ROOT

print('Project root:', PROJECT_ROOT)
print('Evaluation workspace:', THESIS_ROOT)
print('Original dataset root:', ORIGINAL_DATASET_ROOT)
print('Preferred dataset root:', PREFERRED_DATASET_ROOT)
print('Dataset root:', DATASET_ROOT)
print('Torch version:', torch.__version__)
print('Sklearn available for ROC/PR plots:', SKLEARN_AVAILABLE)


Project root: /home/basudeo/Documents/Thesis
Evaluation workspace: /home/basudeo/Documents/Thesis/04_model_evaluation
Original dataset root: /home/basudeo/Documents/Thesis/03_dataset/hybrid_maneuvers_dataset
Preferred dataset root: /home/basudeo/Documents/Thesis/03_dataset/hybrid_maneuvers_dataset
Dataset root: /home/basudeo/Documents/Thesis/03_dataset/hybrid_maneuvers_dataset
Torch version: 2.3.1+cu118
Sklearn available for ROC/PR plots: True


In [3]:
# Shared experiment configuration.
# Keep this block explicit and heavily commented because it is the main place
# you will tune once more data has been collected.
LABEL_MODE = 'reduced'      # Use the balanced 5-class setting by default.
SEED = 42                   # Fix the split and initialization for reproducibility.
PAST_LEN = 10               # Number of past timesteps used to predict the current maneuver.
FUTURE_LEN = 5             # Number of future steps used for trajectory targets (ADE/FDE/RMSE).
SCAN_BEAMS = 512            # Number of lidar beams after resampling.
RANGE_CLIP = 30.0           # Maximum lidar range used for normalization.
BATCH_SIZE = 32             # Batch size for trainable models.
EPOCHS = 30                 # Upper bound; early stopping will usually stop sooner.
EARLY_STOPPING_PATIENCE = 5 # Patience on validation macro-F1.
LR = 1e-3                   # Adam learning rate.
WEIGHT_DECAY = 1e-4         # Small regularization on trainable models.
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
MAX_SAMPLES = None          # Set to an integer for quick smoke tests.
USE_CPU = False             # Force CPU if needed.

# Hidden sizes for the current classification baselines.
CNN_HIDDEN = 96
GRAPH_HIDDEN = 96
FUSION_HIDDEN = 128
LSTM_HIDDEN = 128
LSTM_LAYERS = 1
MSG_PASSES = 2
DROPOUT = 0.10
TRANSFORMER_HEADS = 4
TRANSFORMER_LAYERS = 2
TRANSFORMER_FF = 256

# Device selection is kept simple and transparent.
device = torch.device('cuda' if torch.cuda.is_available() and not USE_CPU else 'cpu')
print('Device:', device)


Device: cuda


In [4]:
# Shared data and evaluation utilities live in one helper module so that
# all notebooks reuse the same dataset, split, path-remapping, and trajectory metrics.
from dataset_helper import (
    build_label_mapping,
    build_run_manifest,
    build_sample_table,
    compute_trajectory_metrics,
    configure_helper,
    group_streams,
    prepare_result_dirs,
    save_mean_step_error_plot,
    save_or_load_fixed_split,
    save_run_manifest,
    save_trajectory_overlay_plots,
    set_seed,
    timestamp_tag,
)

configure_helper(
    dataset_root=DATASET_ROOT,
    original_dataset_root=ORIGINAL_DATASET_ROOT,
    results_root=RESULTS_ROOT,
    weights_root=WEIGHTS_ROOT,
)


In [5]:
# Resolve the project structure relative to this notebook.
PROJECT_ROOT = Path.cwd().resolve().parent
EVAL_ROOT = PROJECT_ROOT
RESULTS_ROOT = EVAL_ROOT / 'results'
WEIGHTS_ROOT = EVAL_ROOT / 'model_weights'
SPLITS_ROOT = EVAL_ROOT / 'splits'

ORIGINAL_DATASET_ROOT = PROJECT_ROOT.parent / '03_dataset' / 'husky_control_dataset'
PREFERRED_DATASET_ROOT = ORIGINAL_DATASET_ROOT
DATASET_ROOT = PREFERRED_DATASET_ROOT

print('Original dataset root:', ORIGINAL_DATASET_ROOT)
print('Preferred dataset root:', PREFERRED_DATASET_ROOT)
print('Dataset root:', DATASET_ROOT)


Original dataset root: /home/basudeo/Documents/Thesis/03_dataset/husky_control_dataset
Preferred dataset root: /home/basudeo/Documents/Thesis/03_dataset/husky_control_dataset
Dataset root: /home/basudeo/Documents/Thesis/03_dataset/husky_control_dataset


In [6]:
set_seed(SEED)
labels, label_mapping = build_label_mapping(LABEL_MODE)
streams = group_streams(DATASET_ROOT, allowed_labels=None, label_mapping=None)
sample_table = build_sample_table(streams, past_len=PAST_LEN, future_len=FUTURE_LEN)
split_path = SPLITS_ROOT / f'husky_control_split_{LABEL_MODE}_seed{SEED}_past{PAST_LEN}_future{FUTURE_LEN}.json'
split_info = save_or_load_fixed_split(sample_table, split_path, SEED, TRAIN_RATIO, VAL_RATIO, PAST_LEN, FUTURE_LEN)
assert split_info['sample_count'] == len(sample_table), (
    f"Split file {split_path} does not match the current dataset: "
    f"split has {split_info['sample_count']} samples, current table has {len(sample_table)}"
)

print('Controller-state metadata labels retained for plot titles only:', labels)
print('Split path:', split_path)
print('Total samples in canonical table:', len(sample_table))
print('Train / Val / Test:', len(split_info['train_indices']), len(split_info['val_indices']), len(split_info['test_indices']))
print('Future horizon:', split_info['future_len'])


Controller-state metadata labels retained for plot titles only: ['go_to_goal', 'avoid_left', 'avoid_right', 'commit_forward', 'arrived']
Split path: /home/basudeo/Documents/Thesis/04_model_evaluation/splits/husky_control_split_reduced_seed42_past10_future5.json
Total samples in canonical table: 9012
Train / Val / Test: 6308 1351 1353
Future horizon: 5


In [7]:
MODEL_SLUG = 'cv_baseline'
TIMESTAMP = timestamp_tag()
result_dir, weight_dir, plot_dir = prepare_result_dirs(MODEL_SLUG)

# Constant-velocity baseline:
# use the final observed planar velocity and extrapolate the future trajectory
# in the world frame over the saved future timestamps.
test_indices = split_info['test_indices']
targets = np.asarray([labels.index(row['raw_label']) if row['raw_label'] in labels else 0 for row in (sample_table[idx] for idx in test_indices)], dtype=np.int64)

all_pred_future_xy = []
all_true_future_xy = []
for sample_idx in test_indices:
    meta = sample_table[sample_idx]
    stream = streams[meta['stream_idx']]
    anchor = stream[meta['anchor_index']]
    state = anchor['state']
    vx = float(state.get('vx', 0.0))
    vy = float(state.get('vy', 0.0))
    future_dt = np.asarray(meta['future_dt'], dtype=np.float32)
    pred_future_xy = np.stack([vx * future_dt, vy * future_dt], axis=-1)
    all_pred_future_xy.append(pred_future_xy)
    all_true_future_xy.append(np.asarray(meta['future_xy'], dtype=np.float32))

pred_future_xy = np.stack(all_pred_future_xy, axis=0)
true_future_xy = np.stack(all_true_future_xy, axis=0)
metrics = compute_trajectory_metrics(pred_future_xy, true_future_xy)
metrics['split_path'] = str(split_path)

run_manifest = build_run_manifest(
    model_slug=MODEL_SLUG,
    timestamp=TIMESTAMP,
    labels=labels,
    split_path=split_path,
    extra={
        'task': 'trajectory_only',
        'baseline_type': 'constant_velocity',
        'future_len': FUTURE_LEN,
        'trajectory_baseline': 'last_observed_velocity_world_frame',
    },
)
save_run_manifest(result_dir, run_manifest, TIMESTAMP)
(result_dir / 'latest_run_manifest.json').write_text(json.dumps(run_manifest, indent=2))

overlay_paths = save_trajectory_overlay_plots(
    pred_future_xy,
    true_future_xy,
    targets,
    labels,
    plot_dir,
    prefix=TIMESTAMP,
    max_plots=8,
)
step_error_path = save_mean_step_error_plot(
    pred_future_xy,
    true_future_xy,
    plot_dir / f'mean_step_error_{TIMESTAMP}.png',
    title=f'{MODEL_SLUG} Mean Future-Step Error',
)
prediction_export_path = result_dir / f'trajectory_predictions_{TIMESTAMP}.npz'
np.savez_compressed(
    prediction_export_path,
    sample_ids=np.asarray([sample_table[idx]['sample_id'] for idx in test_indices], dtype=str),
    pred_future_xy=pred_future_xy,
    true_future_xy=true_future_xy,
)
latest_prediction_export_path = result_dir / 'latest_trajectory_predictions.npz'
np.savez_compressed(
    latest_prediction_export_path,
    sample_ids=np.asarray([sample_table[idx]['sample_id'] for idx in test_indices], dtype=str),
    pred_future_xy=pred_future_xy,
    true_future_xy=true_future_xy,
)

metrics['trajectory_overlay_plots'] = overlay_paths
metrics['mean_step_error_plot'] = step_error_path
metrics['trajectory_prediction_file'] = str(latest_prediction_export_path)
metrics['status'] = 'saved'

metrics_path = result_dir / f'metrics_{TIMESTAMP}.json'
metrics_path.write_text(json.dumps(metrics, indent=2))
(result_dir / 'latest_metrics.json').write_text(json.dumps(metrics, indent=2))

print('CV trajectory baseline metrics:')
print(json.dumps(metrics, indent=2))


CV trajectory baseline metrics:
{
  "ADE": 17.986391067504883,
  "FDE": 27.877450942993164,
  "RMSE": 56.98990249633789,
  "split_path": "/home/basudeo/Documents/Thesis/04_model_evaluation/splits/husky_control_split_reduced_seed42_past10_future5.json",
  "trajectory_overlay_plots": [
    "/home/basudeo/Documents/Thesis/04_model_evaluation/results/cv_baseline/plots/20260421_195142_trajectory_overlay_00.png",
    "/home/basudeo/Documents/Thesis/04_model_evaluation/results/cv_baseline/plots/20260421_195142_trajectory_overlay_01.png",
    "/home/basudeo/Documents/Thesis/04_model_evaluation/results/cv_baseline/plots/20260421_195142_trajectory_overlay_02.png",
    "/home/basudeo/Documents/Thesis/04_model_evaluation/results/cv_baseline/plots/20260421_195142_trajectory_overlay_03.png",
    "/home/basudeo/Documents/Thesis/04_model_evaluation/results/cv_baseline/plots/20260421_195142_trajectory_overlay_04.png",
    "/home/basudeo/Documents/Thesis/04_model_evaluation/results/cv_baseline/plots/202